# Trotter Time Evolution (Braket)

Evolve a state under a ZZ + transverse field Hamiltonian using first-order Trotter decomposition built from Braket gates, compared against exact evolution.

In [ ]:
import numpy as np
from braket.circuits import Circuit, ResultType
from braket.devices import LocalSimulator

## Hamiltonian and utilities

In [ ]:
I2 = np.eye(2, dtype=complex)
PAULI_X = np.array([[0, 1], [1, 0]], dtype=complex)
PAULI_Z = np.array([[1, 0], [0, -1]], dtype=complex)

def pauli_string(ops):
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result

J, H_FIELD = 1.0, 0.5
H_MATRIX = (
    J * pauli_string([PAULI_Z, PAULI_Z])
    + H_FIELD * pauli_string([PAULI_X, I2])
    + H_FIELD * pauli_string([I2, PAULI_X])
)

def exact_unitary(t):
    eigenvalues, eigenvectors = np.linalg.eigh(H_MATRIX)
    exp_diag = np.exp(-1j * eigenvalues * t)
    return eigenvectors @ np.diag(exp_diag) @ eigenvectors.conj().T

def simulate(circuit):
    circuit.add_result_type(ResultType.StateVector())
    device = LocalSimulator()
    task = device.run(circuit, shots=0)
    return np.array(task.result().result_types[0].value, dtype=complex)

## Trotter circuit

In [ ]:
def trotter_circuit(t, n_steps):
    dt = t / n_steps
    circuit = Circuit()
    circuit.x(1)  # |01⟩
    for _ in range(n_steps):
        circuit.cnot(0, 1)
        circuit.rz(1, 2.0 * J * dt)
        circuit.cnot(0, 1)
        circuit.rx(0, 2.0 * H_FIELD * dt)
        circuit.rx(1, 2.0 * H_FIELD * dt)
    return circuit

## Compare exact vs Trotter

In [ ]:
psi0 = np.zeros(4, dtype=complex)
psi0[1] = 1.0  # |01⟩
times = [0.5, 1.0, 2.0, 5.0]
n_trotter = 20

for t in times:
    psi_exact = exact_unitary(t) @ psi0
    psi_trotter = simulate(trotter_circuit(t, n_trotter))
    fidelity = float(np.abs(np.dot(psi_exact.conj(), psi_trotter)) ** 2)
    exact_p = np.abs(psi_exact) ** 2
    trotter_p = np.abs(psi_trotter) ** 2
    print(f"t={t:.1f}  exact: {[f'{p:.4f}' for p in exact_p]}  Trotter: {[f'{p:.4f}' for p in trotter_p]}  fidelity: {fidelity:.6f}")

## Trotter convergence

In [ ]:
t_fixed = 2.0
psi_exact_final = exact_unitary(t_fixed) @ psi0

for n_steps in [1, 2, 5, 10, 20, 50]:
    psi_approx = simulate(trotter_circuit(t_fixed, n_steps))
    fidelity = float(np.abs(np.dot(psi_exact_final.conj(), psi_approx)) ** 2)
    dist = float(np.linalg.norm(psi_exact_final - psi_approx))
    print(f"  steps={n_steps:>3d}  fidelity={fidelity:.6f}  dist={dist:.8f}")